In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install transformers datasets scikit-learn openpyxl -q

In [3]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)

from sklearn.metrics import roc_auc_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [5]:
train = pd.read_excel("toxic_labeled.xlsx")
test  = pd.read_excel("toxic_no_label_evaluation.xlsx")

print("Train shape:", train.shape)
print("Label distribution:\n", train['label'].value_counts())

Train shape: (9000, 2)
Label distribution:
 label
0    4506
1    4494
Name: count, dtype: int64


In [6]:
# Drop null text rows
train = train.dropna(subset=['text']).reset_index(drop=True)
test['text'] = test['text'].fillna('')

print("After cleaning — Train shape:", train.shape)

TEXT_COL  = 'text'
LABEL_COL = 'label'
y = train[LABEL_COL].values

After cleaning — Train shape: (8996, 2)


In [7]:
vec = TfidfVectorizer(max_features=10000, ngram_range=(1,2), analyzer='char_wb')
X = vec.fit_transform(train[TEXT_COL])

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
scores = cross_val_score(lr, X, y, cv=5, scoring='roc_auc')
print(f"Baseline CV ROC-AUC: {scores.mean():.4f}")

# Save baseline
lr.fit(X, y)
baseline_preds = (lr.predict_proba(vec.transform(test[TEXT_COL]))[:, 1] >= 0.5).astype(int)
test_baseline = pd.read_excel("toxic_no_label_evaluation.xlsx")
test_baseline['label'] = baseline_preds
test_baseline.to_excel("toxic_no_label_baseline.xlsx", index=False)
print("✅ Baseline saved")

Baseline CV ROC-AUC: 0.9393
✅ Baseline saved


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

MODEL = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True,
                     padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(
    train[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: 'label'})
)
train_ds = train_ds.map(tokenize, batched=True)
split = train_ds.train_test_split(test_size=0.1, seed=42)
print("✅ Dataset ready")
print("Train size:", len(split['train']))
print("Val size:", len(split['test']))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8996 [00:00<?, ? examples/s]

✅ Dataset ready
Train size: 8096
Val size: 900


In [9]:
#  Model Training

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

args = TrainingArguments(
    output_dir="./c3_results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    seed=42,
    logging_steps=50,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
)
trainer.train()

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.241933,0.163803
2,0.178917,0.171549
3,0.141125,0.158520


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1518, training_loss=0.21988185191814136, metrics={'train_runtime': 464.5372, 'train_samples_per_second': 52.284, 'train_steps_per_second': 3.268, 'total_flos': 1597610328145920.0, 'train_loss': 0.21988185191814136, 'epoch': 3.0})

In [12]:
# ── 8. Evaluate ROC-AUC (SCREENSHOT THIS) ────────────────
val_preds = trainer.predict(split['test'])
val_proba = torch.softmax(
    torch.tensor(val_preds.predictions), dim=1
).numpy()[:, 1]
val_true = np.array(split['test']['label'])

roc_auc = roc_auc_score(val_true, val_proba)
val_labels_pred = (val_proba >= 0.5).astype(int)

print("\n" + "="*50)
print("VALIDATION RESULTS —")
print("="*50)
print(classification_report(val_true, val_labels_pred,
      target_names=['Non-Toxic (0)', 'Toxic (1)']))
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("="*50)


VALIDATION RESULTS —
               precision    recall  f1-score   support

Non-Toxic (0)       0.97      0.96      0.96       444
    Toxic (1)       0.96      0.97      0.96       456

     accuracy                           0.96       900
    macro avg       0.96      0.96      0.96       900
 weighted avg       0.96      0.96      0.96       900

ROC-AUC Score: 0.9922


In [11]:
# ── 9. Predict & Save Final Submission ───────────────────
test_ds = Dataset.from_pandas(test[[TEXT_COL]])
test_ds = test_ds.map(tokenize, batched=True)

final_preds = trainer.predict(test_ds)
final_proba = torch.softmax(
    torch.tensor(final_preds.predictions), dim=1
).numpy()[:, 1]
final_labels = (final_proba >= 0.5).astype(int)

# Fill label column
test_final = pd.read_excel("toxic_no_label_evaluation.xlsx")
test_final['label'] = final_labels

# Verify
print("Predicted label values:", pd.Series(final_labels).value_counts().to_dict())
print("Expected: 0 / 1")
print("Null labels:", test_final['label'].isnull().sum())
print("Total rows:", len(test_final))

test_final.to_excel("toxic_no_label_evaluation.xlsx", index=False)
print("✅ Final submission saved — toxic_no_label_evaluation.xlsx")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Predicted label values: {1: 512, 0: 488}
Expected: 0 / 1
Null labels: 0
Total rows: 1000
✅ Final submission saved — toxic_no_label_evaluation.xlsx


In [13]:
from google.colab import files
files.download("toxic_no_label_evaluation.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>